# AI Move Survey on a free Colab GPU

Runs the whole demo here (room survey AI, "Talk to AI" chat, leads, and the website) and prints one public link.
Your laptop can be off.

**Each time:** **Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**, allow Google
Drive access when asked, and wait about 10 minutes. The newest code is downloaded from GitHub every time,
so after pushing a change you only need to run it again.
The link appears at the bottom. Keep this tab open: free Colab stops after about 90 minutes without
activity and after about 12 hours at most, and the link changes every session.

Leads are saved on Colab's disk and copied to `My Drive/ai-room-survey/survey.db` every minute, so
they survive between sessions.

In [ ]:
#@title 1. Settings
ADMIN_PASSWORD = ""  #@param {type:"string"}
CHAT_MODEL = "qwen3:4b"  #@param ["qwen3:4b", "qwen3:1.7b"]
DRIVE_FOLDER = "ai-room-survey"  #@param {type:"string"}
REPO_URL = "https://github.com/tusharjain2001/moversely.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import secrets
if not ADMIN_PASSWORD:
    ADMIN_PASSWORD = secrets.token_urlsafe(9)
print("Password for the /admin leads page:", ADMIN_PASSWORD)

In [ ]:
#@title 2. Check the GPU and connect Google Drive
import os, subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "No GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Run all."
print("GPU:", gpu)

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Leads will be backed up to:", DRIVE_DIR)

In [ ]:
#@title 3. Download the newest code from GitHub and install the AI libraries (about 3 minutes)
!rm -rf /content/app && git clone -q --depth 1 --branch "$BRANCH" "$REPO_URL" /content/app
!cd /content/app && git log -1 --format="Code version: %h %s (%cd)" --date=short
!pip install -q "transformers==5.20.1" "timm==1.0.30" "git+https://github.com/microsoft/MoGe.git" "sqlalchemy==2.0.55" httpx "fastapi[standard]" python-multipart
print("Done")

In [ ]:
#@title 4. Install the chat AI (Ollama + Qwen3, about 2 minutes)
import time, urllib.request

!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /content/ollama-install.log 2>&1

def wait_for(url, seconds, label):
    for _ in range(seconds):
        try:
            urllib.request.urlopen(url, timeout=3)
            print(label, "is up")
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"{label} did not start; see the logs in /content")

ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                          env=dict(os.environ, OLLAMA_KEEP_ALIVE="24h"))
wait_for("http://127.0.0.1:11434/api/version", 60, "Ollama")
!ollama pull $CHAT_MODEL
# On the T4 the chat model can use the GPU, so no CPU-only setting here.
open("/content/Modelfile", "w").write(f"FROM {CHAT_MODEL}\nPARAMETER num_ctx 4096\nPARAMETER temperature 0.2\n")
!ollama create survey-chat -f /content/Modelfile

In [ ]:
#@title 5. Build the website (about 3 minutes)
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get install -y -qq nodejs > /dev/null
!node -v
!cd /content/app/web && npm ci --no-audit --no-fund --loglevel=error && npx next build 2>&1 | tail -15

In [ ]:
#@title 6. Start the AI server (first time: about 3 minutes to download the models)
import shutil, sqlite3, threading

LOCAL_DB = "/content/survey.db"
DRIVE_DB = f"{DRIVE_DIR}/survey.db"
if os.path.exists(DRIVE_DB):
    shutil.copy(DRIVE_DB, LOCAL_DB)  # leads from earlier sessions
    print("Restored leads from Google Drive")

with open("/content/app/ai/.env", "w") as f:
    f.write(f"ADMIN_PASSWORD={ADMIN_PASSWORD}\n")
    f.write(f"DATABASE_URL=sqlite:///{LOCAL_DB}\n")
    f.write("CHAT_AI_MODEL=survey-chat\n")
    f.write("TRANSFORMERS_VERBOSITY=error\n")

ai_server = subprocess.Popen(["python", "server_local.py"], cwd="/content/app/ai",
                             stdout=open("/content/ai-server.log", "w"), stderr=subprocess.STDOUT)
wait_for("http://127.0.0.1:8000/health", 900, "AI server")


def back_up_leads():
    """Copy the leads database to Google Drive every minute."""
    while True:
        time.sleep(60)
        try:
            if os.path.exists(LOCAL_DB):
                with sqlite3.connect(LOCAL_DB) as src, sqlite3.connect(DRIVE_DB + ".tmp") as dst:
                    src.backup(dst)
                os.replace(DRIVE_DB + ".tmp", DRIVE_DB)
        except Exception as e:
            print("Leads backup failed:", e)


threading.Thread(target=back_up_leads, daemon=True).start()

In [ ]:
#@title 7. Start the website and the public link
website = subprocess.Popen(["npx", "next", "start", "-p", "3000"], cwd="/content/app/web",
                           stdout=open("/content/website.log", "w"), stderr=subprocess.STDOUT)
wait_for("http://127.0.0.1:3000/", 120, "Website")

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:3000"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
import re
link = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        link = m.group(0)
        break
assert link, "No public link yet; see /content/tunnel.log"
wait_for(link + "/ai/health", 60, "Public link")
print()
print("=" * 60)
print("  Website:     ", link)
print("  Talk to AI:  ", link + "/chat")
print("  Leads page:  ", link + "/admin", "  password:", ADMIN_PASSWORD)
print("=" * 60)
print("Keep this tab open while people are testing.")

In [ ]:
#@title 8. (Optional) See what the servers are doing
!tail -n 15 /content/ai-server.log
!tail -n 5 /content/website.log